# NHS A&E Monthly Source Data Profile

## Purpose

This notebook performs reproducible structural and data-quality profiling of
the raw NHS England Monthly A&E source used by the project.

The notebook does not clean, transform, overwrite, or otherwise modify the
raw source file.

## Source

- **Publisher:** NHS England
- **Dataset:** Monthly A&E September 2026
- **Reporting period:** September 2026
- **Local raw file:** `../data/raw/nhs_ae_monthly_2026_09.csv`

## Profiling objectives

1. Verify source provenance and file availability.
2. Validate the 22-column source schema.
3. Review inferred data types.
4. Evaluate missingness.
5. Separate provider records from the published `TOTAL` control record.
6. Test the proposed provider business key.
7. Identify negative values in activity measures.
8. Reconcile provider-level sums against the published `TOTAL` record.
9. Produce an explicit source-quality gate before transformation.

In [ ]:
from pathlib import Path

import pandas as pd

from nhs_ae.ingestion.checksums import calculate_sha256
from nhs_ae.validation.profiling import (
    build_source_profile,
    provider_records,
    reconcile_published_totals,
    total_records,
)
from nhs_ae.validation.schema import (
    IDENTIFIER_COLUMNS,
    MEASURE_COLUMNS,
    PROVIDER_KEY_COLUMNS,
    REQUIRED_COLUMNS,
    find_missing_columns,
    find_unexpected_columns,
    has_expected_column_order,
)


In [ ]:
PROJECT_ROOT = Path.cwd().parent

RAW_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "nhs_ae_monthly_2026_09.csv"
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw source: {RAW_FILE}")
print(f"Source exists: {RAW_FILE.exists()}")

In [ ]:
file_size_bytes = RAW_FILE.stat().st_size
file_sha256 = calculate_sha256(RAW_FILE)

source_integrity = pd.DataFrame(
    {
        "property": [
            "File name",
            "File size (bytes)",
            "SHA-256",
        ],
        "value": [
            RAW_FILE.name,
            file_size_bytes,
            file_sha256,
        ],
    }
)

source_integrity

In [ ]:
df = pd.read_csv(RAW_FILE)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")

df.head()

## 1. Schema validation

The source schema is compared with the controlled schema contract defined in
`nhs_ae.validation.schema`.

The validation checks:

- missing required columns;
- unexpected columns;
- source column order;
- expected identifier and measure counts.

A schema change is treated as a condition requiring investigation before
downstream processing continues.

In [ ]:
source_columns = df.columns.tolist()

missing_columns = find_missing_columns(source_columns)
unexpected_columns = find_unexpected_columns(source_columns)
expected_order = has_expected_column_order(source_columns)

schema_summary = pd.DataFrame(
    {
        "check": [
            "Required columns",
            "Identifier columns",
            "Measure columns",
            "Missing required columns",
            "Unexpected columns",
            "Expected column order",
        ],
        "result": [
            len(REQUIRED_COLUMNS),
            len(IDENTIFIER_COLUMNS),
            len(MEASURE_COLUMNS),
            len(missing_columns),
            len(unexpected_columns),
            expected_order,
        ],
    }
)

schema_summary

In [ ]:
schema_exceptions = pd.DataFrame(
    {
        "exception_type": [
            "Missing columns",
            "Unexpected columns",
        ],
        "columns": [
            ", ".join(missing_columns) or "None",
            ", ".join(unexpected_columns) or "None",
        ],
    }
)

schema_exceptions


## 2. Structural and data-quality profile

Provider-level records are analysed separately from the published `TOTAL`
control record.

The profile evaluates record counts, organisation coverage, completeness,
candidate-key uniqueness, and non-negative activity measures.

Cell 8: Structural and quality profile

In [ ]:
profile = build_source_profile(df)

profile_summary = pd.DataFrame(
    {
        "metric": [
            "Total rows",
            "Total columns",
            "Provider rows",
            "Published TOTAL rows",
            "Unique organisation codes",
            "Missing values",
            "Duplicate provider keys",
            "Negative measure values",
        ],
        "value": [
            profile.row_count,
            profile.column_count,
            profile.provider_row_count,
            profile.total_row_count,
            profile.unique_org_count,
            profile.missing_value_count,
            profile.duplicate_provider_key_count,
            profile.negative_measure_count,
        ],
    }
)

profile_summary

In [ ]:
missingness = (
    df.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

missingness["missing_percentage"] = (
    missingness["missing_count"]
    .div(len(df))
    .mul(100)
)

missingness

## 3. Data types

Pandas-inferred source data types are reviewed before any explicit type
conversion is introduced into the transformation pipeline.

In [ ]:
data_types = (
    df.dtypes
    .astype(str)
    .rename("pandas_dtype")
    .to_frame()
)

data_types

## 4. Provider and published TOTAL records

The published `TOTAL` row is retained as a control record for reconciliation.
It is excluded from provider-level analysis to avoid double counting.

In [ ]:
providers = provider_records(df)
published_totals = total_records(df)

print(f"Provider records: {len(providers):,}")
print(
    "Published TOTAL records: "
    f"{len(published_totals):,}"
)

published_totals[
    list(IDENTIFIER_COLUMNS)
]

## 5. Published-total reconciliation

Provider-level values are aggregated across each of the 18 controlled
numeric measures and compared with the published `TOTAL` control record.

A difference of zero indicates that the calculated provider aggregation
reconciles to the published control value for that measure.

In [ ]:
reconciliation = reconcile_published_totals(df)

reconciliation

In [ ]:
reconciliation_summary = pd.DataFrame(
    {
        "metric": [
            "Measures tested",
            "Measures reconciled",
            "Measures not reconciled",
            "Maximum absolute difference",
        ],
        "value": [
            len(reconciliation),
            int(reconciliation["reconciled"].sum()),
            int((~reconciliation["reconciled"]).sum()),
            reconciliation["difference"].abs().max(),
        ],
    }
)

reconciliation_summary

## 6. Provider business-key validation

The candidate provider-level business key is:

`Period + Org Code`

The published `TOTAL` control record is excluded before provider-level
uniqueness is assessed.

In [ ]:
duplicate_provider_keys = providers.duplicated(
    subset=list(PROVIDER_KEY_COLUMNS),
    keep=False,
)

duplicate_key_records = providers.loc[
    duplicate_provider_keys,
    list(PROVIDER_KEY_COLUMNS),
]

print(
    "Duplicate provider-key records:",
    len(duplicate_key_records),
)

duplicate_key_records

## 7. Source quality gate

The raw source is approved for downstream transformation only when all
critical structural, completeness, uniqueness, validity, and reconciliation
checks pass.

A failed critical check prevents the source from proceeding into the
transformation pipeline until the issue has been investigated.

In [ ]:
quality_checks = {
    "required_columns_present": not missing_columns,
    "no_unexpected_columns": not unexpected_columns,
    "expected_column_order": expected_order,
    "exactly_one_total_record": (
        profile.total_row_count == 1
    ),
    "no_missing_values": (
        profile.missing_value_count == 0
    ),
    "unique_provider_keys": (
        profile.duplicate_provider_key_count == 0
    ),
    "non_negative_measures": (
        profile.negative_measure_count == 0
    ),
    "published_totals_reconcile": bool(
        reconciliation["reconciled"].all()
    ),
}

quality_gate = pd.DataFrame(
    {
        "check": quality_checks.keys(),
        "passed": quality_checks.values(),
    }
)

quality_gate

## 8. Quality-gate decision

In [ ]:
source_quality_passed = all(quality_checks.values())

print(
    "SOURCE QUALITY GATE:",
    "PASS" if source_quality_passed else "FAIL",
)

In [ ]:
assert source_quality_passed, (
    "Critical source-quality checks failed. "
    "Do not proceed to transformation."
)

## 9. Profiling conclusion

The September 2026 monthly NHS A&E source passed the project's initial
source-quality gate.

### Data-quality findings

- The source conforms to the controlled 22-column schema.
- The schema contains 4 identifier fields and 18 controlled measure fields.
- Provider records are distinguishable from the published `TOTAL` control
  record.
- No missing values were identified by the controlled checks.
- No duplicate provider business keys were identified.
- No negative values were identified across the controlled measure columns.
- Provider-level aggregations reconcile against the published `TOTAL` control
  record across all 18 controlled measures.

### Analytical treatment

The published `TOTAL` record will not be treated as a provider in downstream
provider-level analysis. The record will instead be retained as a
reconciliation control.

The candidate provider-level business key is:

`Period + Org Code`

### Transformation decision

The source is approved to proceed to controlled transformation, subject to
the validation rules defined by this project.

The raw NHS source has not been cleaned, transformed, or overwritten during
this profiling stage.

# 10. Extended identifier and categorical integrity profiling

The initial source-quality gate established structural completeness,
uniqueness, non-negative measures, and reconciliation to the published
TOTAL control record.

This section performs deeper profiling of the source identifiers before
transformation.

## 10.1 Identifier cardinality

Cardinality represents the number of distinct values present in a field.

The following identifier fields are assessed:

- `Period`
- `Org Code`
- `Parent Org`
- `Org name`

The published `TOTAL` control record is excluded from provider-level
identifier analysis because it is not an operational provider.

In [ ]:
identifier_cardinality = pd.DataFrame(
    {
        "column": IDENTIFIER_COLUMNS,
        "row_count": [
            len(providers)
            for column in IDENTIFIER_COLUMNS
        ],
        "unique_values": [
            providers[column].nunique(dropna=False)
            for column in IDENTIFIER_COLUMNS
        ],
    }
)

identifier_cardinality["duplicate_values"] = (
    identifier_cardinality["row_count"]
    - identifier_cardinality["unique_values"]
)

identifier_cardinality

## 10.2 Parent organisation distribution

Frequency analysis identifies how provider records are distributed across
the values of `Parent Org`.

This is descriptive profiling only. A high or low frequency is not treated
as a data-quality failure without additional evidence.

In [ ]:
parent_org_distribution = (
    providers["Parent Org"]
    .value_counts(
        dropna=False,
    )
    .rename_axis("parent_org")
    .reset_index(name="provider_records")
)

parent_org_distribution

## 10.3 Blank and whitespace identifier checks

An identifier can be technically non-null while still being unusable.

This analysis therefore distinguishes:

- null values;
- empty strings;
- whitespace-only strings;
- values containing leading or trailing whitespace.

In [ ]:
identifier_integrity_rows = []

for column in IDENTIFIER_COLUMNS:
    values = providers[column].astype("string")

    null_count = int(
        values.isna().sum()
    )

    blank_count = int(
        values.fillna("").eq("").sum()
    )

    whitespace_only_count = int(
        values.fillna("")
        .str.fullmatch(r"\s+")
        .fillna(False)
        .sum()
    )

    surrounding_whitespace_count = int(
        values.fillna("")
        .ne(values.fillna("").str.strip())
        .sum()
    )

    identifier_integrity_rows.append(
        {
            "column": column,
            "null_values": null_count,
            "blank_values": blank_count,
            "whitespace_only_values": whitespace_only_count,
            "surrounding_whitespace_values": (
                surrounding_whitespace_count
            ),
        }
    )

identifier_integrity = pd.DataFrame(
    identifier_integrity_rows
)

identifier_integrity

## 10.4 Parent organisation whitespace investigation

The identifier-integrity profile identified surrounding whitespace in
`Parent Org`.

The following analysis inspects the affected raw values without modifying
the source data. The objective is to determine whether the issue consists
of leading whitespace, trailing whitespace, or both.

In [ ]:
parent_org_values = providers["Parent Org"].astype("string")

parent_org_whitespace_investigation = pd.DataFrame(
    {
        "raw_value": parent_org_values,
        "trimmed_value": parent_org_values.str.strip(),
        "raw_length": parent_org_values.str.len(),
        "trimmed_length": (
            parent_org_values.str.strip().str.len()
        ),
        "has_leading_whitespace": (
            parent_org_values.str.len()
            > parent_org_values.str.lstrip().str.len()
        ),
        "has_trailing_whitespace": (
            parent_org_values.str.len()
            > parent_org_values.str.rstrip().str.len()
        ),
    }
)

parent_org_whitespace_investigation[
    parent_org_whitespace_investigation[
        "raw_value"
    ].ne(
        parent_org_whitespace_investigation[
            "trimmed_value"
        ]
    )
].head(20)


Add a summary cell

In [ ]:
leading_whitespace_count = int(
    parent_org_whitespace_investigation[
        "has_leading_whitespace"
    ].sum()
)

trailing_whitespace_count = int(
    parent_org_whitespace_investigation[
        "has_trailing_whitespace"
    ].sum()
)

both_sides_count = int(
    (
        parent_org_whitespace_investigation[
            "has_leading_whitespace"
        ]
        & parent_org_whitespace_investigation[
            "has_trailing_whitespace"
        ]
    ).sum()
)

parent_org_whitespace_summary = pd.DataFrame(
    {
        "check": [
            "Leading whitespace",
            "Trailing whitespace",
            "Whitespace on both sides",
        ],
        "affected_records": [
            leading_whitespace_count,
            trailing_whitespace_count,
            both_sides_count,
        ],
    }
)

parent_org_whitespace_summary

Compare categories before and after trimming

In [ ]:
parent_org_category_comparison = pd.DataFrame(
    {
        "metric": [
            "Raw distinct Parent Org values",
            "Trimmed distinct Parent Org values",
        ],
        "value": [
            parent_org_values.nunique(
                dropna=False
            ),
            parent_org_values.str.strip().nunique(
                dropna=False
            ),
        ],
    }
)

parent_org_category_comparison

## 10.5 Organisation mapping consistency

This section tests the internal consistency of relationships between provider
identifiers and organisation labels.

The checks assess whether:

1. each `Org Code` maps to one `Org name`;
2. each `Org name` maps to one `Org Code`;
3. each `Org Code` maps to one trimmed `Parent Org`.

These are single-period consistency checks. They do not establish that
organisation mappings remain unchanged across reporting months.

In [ ]:
org_code_to_name = (
    providers.groupby(
        "Org Code",
        dropna=False,
    )
    .agg(
        distinct_org_names=(
            "Org name",
            "nunique",
        ),
        provider_rows=(
            "Org name",
            "size",
        ),
    )
    .reset_index()
)

inconsistent_code_to_name = org_code_to_name.loc[
    org_code_to_name[
        "distinct_org_names"
    ].gt(1)
]

print(
    "Org Codes mapping to multiple Org names:",
    len(inconsistent_code_to_name),
)

inconsistent_code_to_name

### Interpretation: organisation code-to-name mapping

No organisation codes were associated with multiple organisation names in
the September 2026 provider-level source.

This indicates that the relationship:

`Org Code → Org name`

is internally consistent within the current reporting period.

The result does not establish longitudinal consistency. Organisation-code and
organisation-name relationships must be reassessed after multiple monthly
sources have been combined.

In [ ]:
org_name_to_code = (
    providers.groupby(
        "Org name",
        dropna=False,
    )
    .agg(
        distinct_org_codes=(
            "Org Code",
            "nunique",
        ),
        provider_rows=(
            "Org Code",
            "size",
        ),
    )
    .reset_index()
)

inconsistent_name_to_code = org_name_to_code.loc[
    org_name_to_code[
        "distinct_org_codes"
    ].gt(1)
]

print(
    "Org names mapping to multiple Org Codes:",
    len(inconsistent_name_to_code),
)

inconsistent_name_to_code

In [ ]:
provider_mapping_profile = providers.assign(
    parent_org_trimmed=(
        providers["Parent Org"]
        .astype("string")
        .str.strip()
    )
)

org_code_to_parent = (
    provider_mapping_profile.groupby(
        "Org Code",
        dropna=False,
    )
    .agg(
        distinct_parent_orgs=(
            "parent_org_trimmed",
            "nunique",
        ),
        provider_rows=(
            "parent_org_trimmed",
            "size",
        ),
    )
    .reset_index()
)

inconsistent_code_to_parent = (
    org_code_to_parent.loc[
        org_code_to_parent[
            "distinct_parent_orgs"
        ].gt(1)
    ]
)

print(
    "Org Codes mapping to multiple Parent Orgs:",
    len(inconsistent_code_to_parent),
)

inconsistent_code_to_parent

In [ ]:
mapping_consistency_summary = pd.DataFrame(
    {
        "check": [
            "Org Code maps to multiple Org names",
            "Org name maps to multiple Org Codes",
            "Org Code maps to multiple Parent Orgs",
        ],
        "exceptions": [
            len(inconsistent_code_to_name),
            len(inconsistent_name_to_code),
            len(inconsistent_code_to_parent),
        ],
    }
)

mapping_consistency_summary["passed"] = (
    mapping_consistency_summary["exceptions"].eq(0)
)

mapping_consistency_summary

In [ ]:
org_code_pattern_profile = (
    providers["Org Code"]
    .astype("string")
    .str.strip()
    .str.len()
    .value_counts()
    .sort_index()
    .rename_axis("org_code_length")
    .reset_index(name="provider_count")
)

org_code_pattern_profile

## 10.6 Organisation code character profile

This check profiles the character composition of `Org Code`.

The purpose is to identify:

- lowercase characters;
- embedded whitespace;
- characters outside uppercase letters and digits.

This is a profiling check rather than an assumption about the official format
of NHS organisation codes.

In [ ]:
org_code_values = (
    providers["Org Code"]
    .astype("string")
    .str.strip()
)

org_code_character_check = pd.DataFrame(
    {
        "check": [
            "Contains lowercase characters",
            "Contains whitespace",
            "Contains special characters",
        ],
        "affected_records": [
            int(
                org_code_values
                .str.contains(r"[a-z]", regex=True)
                .fillna(False)
                .sum()
            ),
            int(
                org_code_values
                .str.contains(r"\s", regex=True)
                .fillna(False)
                .sum()
            ),
            int(
                (
                    ~org_code_values
                    .str.fullmatch(r"[A-Z0-9]+")
                    .fillna(False)
                ).sum()
            ),
        ],
    }
)

org_code_character_check

## 10.7 Extended identifier-quality summary

This section consolidates the identifier-quality checks performed on the
provider-level source.

The assessment covers:

- null and blank identifiers;
- whitespace-only identifiers;
- organisation code character composition;
- organisation code-to-name consistency;
- organisation name-to-code consistency;
- organisation code-to-parent organisation consistency;
- known source formatting issues.

The purpose is to distinguish between issues that prevent analysis and issues
that can be corrected through a controlled transformation rule.

In [ ]:
extended_identifier_checks = {
    "no_null_identifiers": (
        identifier_integrity["null_values"].sum() == 0
    ),
    "no_blank_identifiers": (
        identifier_integrity["blank_values"].sum() == 0
    ),
    "no_whitespace_only_identifiers": (
        identifier_integrity[
            "whitespace_only_values"
        ].sum() == 0
    ),
    "org_codes_uppercase_alphanumeric": (
        org_code_character_check[
            "affected_records"
        ].sum() == 0
    ),
    "org_code_to_name_consistent": (
        len(inconsistent_code_to_name) == 0
    ),
    "org_name_to_code_consistent": (
        len(inconsistent_name_to_code) == 0
    ),
    "org_code_to_parent_consistent": (
        len(inconsistent_code_to_parent) == 0
    ),
}

extended_identifier_summary = pd.DataFrame(
    {
        "check": extended_identifier_checks.keys(),
        "passed": extended_identifier_checks.values(),
    }
)

extended_identifier_summary

## 10.8 Controlled transformation requirements

Extended profiling identified a formatting issue in the raw `Parent Org`
field.

### Finding

- 167 provider records contain trailing whitespace in `Parent Org`.
- No leading whitespace was identified.
- No records contained whitespace on both sides.
- The raw and trimmed values both produce 7 distinct parent-organisation
  categories.

### Required ETL treatment

During controlled transformation, surrounding whitespace will be removed
from `Parent Org` using:

`str.strip()`

The raw source file will remain unchanged.

### Post-transformation validation

The transformation pipeline must confirm that:

1. surrounding whitespace in the standardised parent-organisation field is zero;
2. provider row count remains unchanged;
3. the number of parent-organisation categories remains consistent unless a
   documented mapping rule requires otherwise;
4. activity measures and published totals remain unchanged;
5. no new null values are introduced.

## 10.9 Extended profiling decision

The extended identifier and categorical profiling identified no critical
provider-identifier failures.

### Checks passed

- No null identifiers were identified.
- No blank identifiers were identified.
- No whitespace-only identifiers were identified.
- Organisation codes contain only uppercase letters and digits in the current
  source.
- `Org Code → Org name` mappings are internally consistent.
- `Org name → Org Code` mappings are internally consistent.
- `Org Code → Parent Org` mappings are internally consistent after temporary
  whitespace trimming.

### Controlled issue identified

`Parent Org` contains trailing whitespace in 167 provider records.

The issue does not prevent use of the source because:

- the issue has been identified and quantified;
- the raw source remains preserved;
- the correction can be applied deterministically during ETL;
- post-transformation validation will verify that the correction does not
  change provider counts, activity measures, or published-total
  reconciliation.

### Decision

**EXTENDED SOURCE PROFILING: PASS WITH CONTROLLED TRANSFORMATION REQUIRED**

The source is suitable to proceed to controlled ETL once the documented
`Parent Org` whitespace rule is implemented and validated.

## 10.10 Extended source-quality gate

The final extended quality gate combines the core source-quality checks with
the additional identifier-integrity checks.

A known formatting issue in `Parent Org` is treated as a controlled ETL
requirement rather than a source-rejection condition because the issue has
been identified, quantified, and assigned a deterministic transformation
rule.


In [ ]:
extended_quality_checks = {
    **quality_checks,
    **extended_identifier_checks,
    "parent_org_whitespace_identified": (
        trailing_whitespace_count == 167
    ),
    "parent_org_trim_preserves_categories": (
        parent_org_values.nunique(dropna=False)
        == parent_org_values.str.strip().nunique(
            dropna=False
        )
    ),
}

extended_quality_gate = pd.DataFrame(
    {
        "check": extended_quality_checks.keys(),
        "passed": extended_quality_checks.values(),
    }
)

extended_quality_gate

## 10.11 Final extended source-profiling decision

The extended quality gate is evaluated programmatically so the source cannot
proceed solely on the basis of visual inspection.

All controlled checks must pass. The identified `Parent Org` trailing
whitespace remains a documented transformation requirement and must be
corrected and revalidated during ETL.

In [ ]:
extended_profile_passed = all(
    extended_quality_checks.values()
)

print(
    "EXTENDED SOURCE PROFILING:",
    (
        "PASS WITH CONTROLLED TRANSFORMATION REQUIRED"
        if extended_profile_passed
        else "FAIL"
    ),
)

In [ ]:
assert extended_profile_passed, (
    "Extended source-profiling checks failed. "
    "Do not proceed to ETL."
)